In [1]:
from ollama import generate
import requests
# chart_supabase_demo.py
import uuid
import matplotlib.pyplot as plt
from datetime import datetime
from supabase import create_client, Client

In [2]:
data_to_upload = {"id": str(uuid.uuid4())}

In [3]:
offers_recieved = {
    "first_offer": {
        "experience_level": "EN",
        "employment_type": "FT",
        "job_title": "Data Scientist",
        "company_location": "FR",
        "company_size": "S",
        "remote_ratio": 100,
        "salary_in_usd": 30000,
    },

    "second_offer": {
        "experience_level": "EN",
        "employment_type": "FT",
        "job_title": "Machine Learning Scientist",
        "company_location": "GB",
        "company_size": "M",
        "remote_ratio": 100,
        "salary_in_usd": 50000,
    },

    "third_offer": {
        "experience_level": "EN",
        "employment_type": "FT",
        "job_title": "Machine Learning Engineer",
        "company_location": "US",
        "company_size": "M",
        "remote_ratio": 100,
        "salary_in_usd": 80000,
    }
}

data_to_upload["offers"] = offers_recieved

In [4]:
predicted_salaries = []

for offer in data_to_upload["offers"].values():
    response = requests.post("https://hosen20-test.hf.space/predict",
                             params = {k: offer[k] for k in list(offer.keys())[:-1]})
    predicted_salaries.append(response.json()["predicted_salary"])

data_to_upload["predicted_salaries"] = predicted_salaries
print(predicted_salaries)

[49277.75, 49277.75, 95021.73913043478]


In [5]:
main_prompt = """
        Write a small sized list of 5 unordered points explaining an offer enclosed in triple backticks,
        use the following metrics encolsed in double backticks,
        add a total score in the end out of 100 according to the evaluated metrics,
        do not add any extra lines, comments, questions, or discussions, just output list and total score,
        just directly start with point 1. with nothing said before it,
        every point out of the 5 should have a brief explanation whether positive or negative.
        In the end show the score by printing Total Score: XX
          """

metrics_prompt = """
        The metrics to use to evaluate the job offer are five:
             1. salary
             2. work culture for each country
             3. taxes by each country
             4. job_title
             5. one other metric of your choice deemed relevant, the last metric should not use remote work
           """

In [6]:
first_offer_analysis = generate("gemma3:1b",
                        f"{main_prompt} \n ```{offers_recieved["first_offer"]}``` \n ``{metrics_prompt}``")
data_to_upload["offer1_explanation"] = first_offer_analysis["response"]
print(data_to_upload["offer1_explanation"])

1. Salary: $30,000 – Relatively low compared to similar roles in the US.
2. Work Culture: France – Generally considered a more formal and structured work environment.
3. Taxes by Country: France – Higher tax rate than many countries, impacting disposable income.
4. Job Title: Data Scientist – Offers a focused and technically demanding role.
5. One Other Metric: Career Growth Potential: Limited compared to some companies.

Total Score: 67



In [7]:
second_offer_analysis = generate("gemma3:1b",
                        f"{main_prompt} \n ```{offers_recieved["second_offer"]}``` \n ``{metrics_prompt}``")
data_to_upload["offer2_explanation"] = second_offer_analysis["response"]
print(data_to_upload["offer2_explanation"])

1.  Salary: $50,000 – Competitively priced within the market.
2.  Work Culture: Positive – Indicates a collaborative and supportive environment.
3.  Taxes by Country: Neutral – Tax rates vary significantly across different locations.
4.  Job Title: Positive – Machine Learning Scientist role is highly desirable.
5.  Company Size: Positive – A mid-sized company offers stability and growth potential.

Total Score: 77


In [8]:
third_offer_analysis = generate("gemma3:1b",
                        f"{main_prompt} \n ```{offers_recieved["third_offer"]}``` \n ``{metrics_prompt}``")
data_to_upload["offer3_explanation"] = third_offer_analysis["response"]
print(data_to_upload["offer3_explanation"])

1.  **Salary:** $80,000 – Slightly below industry average for the role and location.
2.  **Work Culture:** Mixed – Some reports indicate a collaborative but demanding culture, while others suggest a more independent environment.
3.  **Taxes:** $80,000 – Relatively high tax burden in the US, compared to some other countries.
4.  **Job Title:** Machine Learning Engineer – A challenging and potentially competitive role.
5.  **Company Size:** Medium – Offers a good balance between stability and growth potential, but not exceptionally large.

Total Score: 67



In [9]:
data_to_upload["scores"] = [
    int(data_to_upload["offer1_explanation"].split("Total Score: ")[-1][0:2]),
    int(data_to_upload["offer2_explanation"].split("Total Score: ")[-1][0:2]),
    int(data_to_upload["offer3_explanation"].split("Total Score: ")[-1][0:2])
]

print(data_to_upload["scores"])

[67, 77, 67]


In [10]:
# --- Setup Supabase client ---
url = ""
key = ""
supabase: Client = create_client(url, key)

In [11]:
# --- Insert into Supabase ---
def insert_record(record):
    response = supabase.table("analysis").insert(record).execute()
    print("Inserted:", response)

insert_record(data_to_upload)

Inserted: data=[{'id': '5844cb50-2fb0-4e62-9229-d16428222877', 'offers': {'first_offer': {'job_title': 'Data Scientist', 'company_size': 'S', 'remote_ratio': 100, 'salary_in_usd': 30000, 'employment_type': 'FT', 'company_location': 'FR', 'experience_level': 'EN'}, 'third_offer': {'job_title': 'Machine Learning Engineer', 'company_size': 'M', 'remote_ratio': 100, 'salary_in_usd': 80000, 'employment_type': 'FT', 'company_location': 'US', 'experience_level': 'EN'}, 'second_offer': {'job_title': 'Machine Learning Scientist', 'company_size': 'M', 'remote_ratio': 100, 'salary_in_usd': 50000, 'employment_type': 'FT', 'company_location': 'GB', 'experience_level': 'EN'}}, 'predicted_salaries': [49277.75, 49277.75, 95021.73913043478], 'offer1_explanation': '1. Salary: $30,000 – Relatively low compared to similar roles in the US.\n2. Work Culture: France – Generally considered a more formal and structured work environment.\n3. Taxes by Country: France – Higher tax rate than many countries, impact